# Cardiac JEPA — campagne complète (Colab GPU)

Pipeline unique : pré-entraînement JEPA (ViT / CNN / xresnet) → sonde gelée + fine-tuning
(early-stopping) + référence supervisée BN, sur 5 graines, avec macro-AUROC + AUPRC + IC bootstrap.

Tout est **idempotent** : chaque run dont le `result.json` existe déjà est sauté. On peut
relancer après une coupure de session sans rien recalculer (les runs vivent sur le Drive).


## 0 · Configuration (à ajuster)


In [ ]:
REPO_URL   = 'https://github.com/USER/cardiac-jepa.git'  # <-- ton repo
DRIVE_ROOT = '/content/drive/MyDrive/cardiac-jepa'       # runs/ + cache/ persistants ici
BRANCH     = 'main'


## 1 · Drive + repo + dépendances


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.isdir('/content/cardiac-jepa'):
    subprocess.run(['git','clone','-b',BRANCH,REPO_URL,'/content/cardiac-jepa'], check=True)
%cd /content/cardiac-jepa
!git pull --ff-only
!pip -q install wfdb


## 2 · Cache + runs persistants sur le Drive

`cache/ptbxl_100hz.npy` (~1 Go) et `runs/` sont liés au Drive pour survivre aux sessions.
Le cache est construit une fois (nécessite PTB-XL décompressé ; voir `jepa/build_cache.py`).


In [ ]:
import os
for d in ('cache','runs'):
    src = f'{DRIVE_ROOT}/{d}'
    os.makedirs(src, exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): import shutil; shutil.rmtree(d)
        os.symlink(src, d)
print('cache:', os.listdir('cache'))
# Si le cache est vide : décompresser PTB-XL puis  !python -m jepa.build_cache  (voir le module)


## 3 · Plan de la campagne (dry-run)

161 runs : 3 pré-entraînements + 18 sondes + 120 fine-tunings + 20 supervisés.


In [ ]:
!python -m jepa.experiment --dry-run | head -8
!echo ...
!python -m jepa.experiment --dry-run | tail -3


## 4 · Étape A — pré-entraînements JEPA (les 3 backbones)

Le plus long (~100 epochs chacun). `--resume auto` reprend un pré-entraînement interrompu.


In [ ]:
!python -m jepa.experiment --steps pretrain


## 5 · Étape B — grille d'évaluation (sonde + fine-tuning + supervisé)

Relançable à volonté : ne recalcule que ce qui manque. Pour trimmer, ex. `--seeds 0,1,2`
ou `--backbones xresnet`.


In [ ]:
!python -m jepa.experiment --steps probe,finetune,supervised


## 6 · Agrégation → `runs/results.json`

Moyennes ± écart-type inter-graines, écarts pairés JEPA − scratch (t de Student).


In [ ]:
!python -m jepa.aggregate


## 7 · Empaqueter les artefacts LÉGERS pour les figures locales

Zippe uniquement `results.json` + tous les `result.json` + `metrics.csv` (PAS les poids `.pt`).
Quelques centaines de Ko. Télécharge `results_bundle.zip`, décompresse-le à la racine du repo
en local, puis lance `python make_figures.py`.


In [ ]:
import glob, zipfile
light = ['runs/results.json'] + glob.glob('runs/**/result.json', recursive=True) \
        + glob.glob('runs/**/metrics.csv', recursive=True)
light = [p for p in light if os.path.exists(p)]
with zipfile.ZipFile('results_bundle.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in light: z.write(p)
print(f'{len(light)} fichiers ->', round(os.path.getsize('results_bundle.zip')/1024), 'Ko')
from google.colab import files; files.download('results_bundle.zip')


## 8 · (local) Figures + README

`unzip results_bundle.zip` à la racine du repo, puis `python make_figures.py` régénère les
figures (barres d'erreur + IC) ; le README lit les mêmes chiffres. Aucun poids requis.
